# Initial population

This page covers how to say how many people start in each compartment. There are two pieces:

- `InitialPopulation` gives the **totals**: how many people there are in each part of the population;
- `Split` shares a total out between the traits of a property, for example between vaccinated and unvaccinated people.

The totals and the shares can be plain numbers or can depend on the model's parameters. We use the second form to build a population whose vaccinated share is a parameter, so that it can be changed between runs (or calibrated).

Adapted from the [summer2 documentation](https://summer2.readthedocs.io) page `detailed/InitialPopulationGraphobject` at commit `d1537d6188aba85c33c0449b197eef6ad8b03d6c` of [monash-emu/summer2](https://github.com/monash-emu/summer2). Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried and adapted; code is written for summer4.

## A model with age and vaccination

We use a small SIR model stratified by age (young and old) and by vaccination status (`imm`, either `yes` or `no`). The function `build_model` makes the model from an `InitialPopulation`, so we can try several starting populations on the same model. The compartment labels show every combination.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import jax.numpy as jnp

from summer4 import (
    FlowModel,
    InitialPopulation,
    Param,
    Property,
    PropertyMap,
    Split,
    TransitionFlow,
    defer,
)
from summer4.epi import ForceOfInfection

state = Property("state", ("S", "I", "R"))
age = Property("age", ("young", "old"))
imm = Property("imm", ("yes", "no"))
pmap = PropertyMap.from_property(state).stratify(age).stratify(imm)


def build_model(initial):
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["I"],
            ForceOfInfection("foi", infectious=state["I"], contact_rate=0.4),
        )
    )
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
    model.set_initial_population(initial.base, initial.splits)
    return model.compile()


pmap.labels()

## Totals and splits with plain numbers

`InitialPopulation` takes a dictionary. Each key picks out part of the population and each value is the number of people there. Here there are 3,000 young and 7,000 old susceptible people, and 10 infectious people, who are shared between the other groups evenly by default.

A `Split` then shares those people between the traits of one property. `Split(imm, {"yes": 0.2, "no": 0.8})` puts a fifth of every group into the vaccinated trait. The bar chart shows the starting population in each compartment.

In [ ]:
totals = {
    state["S"] & age["young"]: 3000.0,
    state["S"] & age["old"]: 7000.0,
    state["I"]: 10.0,
}
fixed = InitialPopulation(totals, splits=(Split(imm, {"yes": 0.2, "no": 0.8}),))
cm = build_model(fixed)

start = pd.Series(np.asarray(cm.initial_state({}).data), index=pmap.labels(), name="people")
start.plot.bar(
    title="Starting population, fixed 20% vaccinated",
    labels={"index": "compartment", "value": "people"},
)

## A share that depends on a parameter

To make the vaccinated share a parameter, put a `Param` where the number was. `Param("vaccinated")` is looked up by name each time the model runs, so the same compiled model can start from different populations. The unvaccinated weight is one minus the vaccinated share.

The next cell runs the initial population for three values of `vaccinated` and plots the vaccinated susceptible people for each: the bars scale with the parameter.

In [ ]:
with_param = InitialPopulation(
    totals,
    splits=(Split(imm, {"yes": Param("vaccinated"), "no": 1.0 - Param("vaccinated")}),),
)
cm = build_model(with_param)

vaccinated_susceptible = {}
for share in (0.1, 0.5, 0.9):
    y0 = pd.Series(np.asarray(cm.initial_state({"vaccinated": share}).data), index=pmap.labels())
    vaccinated_susceptible[f"vaccinated = {share}"] = y0[
        y0.index.str.contains("state=S") & y0.index.str.contains("imm=yes")
    ]

pd.DataFrame(vaccinated_susceptible).plot.bar(
    barmode="group",
    title="Vaccinated susceptible people for three values of the parameter",
    labels={"index": "compartment", "value": "people", "variable": ""},
)

## A share worked out by your own function

Sometimes the share is not the parameter itself but something worked out from it. To use your own function, wrap it with `defer` and apply it to the parameters: `defer(f)(Param("a"), Param("b"))`. The library calls `f` with the parameter values when the model runs.

Below, vaccination is given as one coverage per age group. The function `coverage_table` turns the two coverages into a table with one row per age group and one column per trait of `imm` (yes, no). Passing `by=(age,)` tells the `Split` that the table has one row for each age group. Inside the function we use `jnp.array`, the array maker from JAX, so the model can still be run quickly. The bars show the susceptible people in each compartment.

In [ ]:
def coverage_table(young, old):
    return jnp.array([[young, 1.0 - young], [old, 1.0 - old]])


by_age = InitialPopulation(
    totals,
    splits=(
        Split(
            imm,
            defer(coverage_table)(Param("coverage_young"), Param("coverage_old")),
            by=(age,),
        ),
    ),
)
cm = build_model(by_age)
params = {"coverage_young": 0.2, "coverage_old": 0.6}

y0 = pd.Series(np.asarray(cm.initial_state(params).data), index=pmap.labels(), name="people")
y0[y0.index.str.contains("state=S")].plot.bar(
    title="Susceptible people by age and vaccination: coverage 20% young, 60% old",
    labels={"index": "compartment", "value": "people"},
)

## Running from this population

The model now starts from the population above and runs like any other. The plot shows the infectious people over time for two coverage settings. Nothing in the model changes between the runs; only the parameter values do.

In [ ]:
runs = {}
settings = {
    "old coverage 0.6": {"coverage_young": 0.2, "coverage_old": 0.6},
    "old coverage 0.1": {"coverage_young": 0.2, "coverage_old": 0.1},
}
for label, p in settings.items():
    result = cm.run(p, t0=0.0, t1=150.0, dt=1.0)
    infectious = result["compartments"].select(state["I"]).sum_by()
    runs[label] = pd.Series(
        np.asarray(infectious.values).reshape(-1),
        index=pd.Index(np.asarray(infectious.times.values), name="day"),
    )

pd.DataFrame(runs).plot(
    title="Infectious people for two vaccination settings",
    labels={"value": "infectious people", "variable": ""},
)

## Things to try

- Change the totals: make the old population 15,000 and see how the starting bars change.
- Add a second `Split` to the tuple, for example one on `age` for the infectious people, with `where=state["I"]`.
- Pass `normalize=True` to a `Split` whose weights do not add up to one and see that the people are still shared out in proportion.
- Change `coverage_table` so that the old are always vaccinated at twice the rate of the young, using only one parameter.